In [ ]:
#GEE LOGISTIC REGRESSEION MODEL 

"""
Sets up environment for per-test GEE analysis
Loads data and defines test direction dictionary
Creates output directory for storing results from each test

CHANGED: every per-test artefact (coefficients, vcov, predictions) now writes
into output_root, not the hardcoded 'geeglm_pertest'. Previously the hc/ic
runs clobbered the 'all' run's predictions.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

#minimal plotting size
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

#load data
data = pd.read_csv('cv_results_all_tests_merged_with_groups.csv')

GROUP = 'ic'   #'all' | 'hc' | 'ic'

def filter_group(df, group):
    if group == 'all':
        out = df.copy()
    elif group == 'hc':
        out = df[df['age_at_ic'].isna() | (df['age_at_test'] < df['age_at_ic'])].copy()
    else:
        out = df[df['age_at_ic'].notna() & (df['age_at_test'] >= df['age_at_ic'])].copy()

    print(f"Group '{group}': {out['eid'].nunique()} people, {len(out)} observations, "
          f"{out.groupby('eid')['dem_all'].first().sum()} with incident dementia")
    return out

data = filter_group(data, GROUP)
output_root = 'geeglm_pertest' if GROUP == 'all' else f'geeglm_pertest_{GROUP}'
os.makedirs(output_root, exist_ok=True)
data = data[data['test_name'] != 'PMem'].copy()

n0 = len(data)
data = data[data['prevalent'] != 1].copy()
print(f"Excluded {n0 - len(data)} post-diagnosis observations; "
      f"{data['eid'].nunique()} people, {len(data)} observations remain")

tests = data['test_name'].unique()

# 1 = higher score is better (invert OR to express "per 1 SD worse")
# -1 = lower score is better (timed tests; OR already reads as "worse")
test_direction = {
    'PMat': -1,
    'RT':   -1,
    'FI':    1,
    'NM':    1,
    'SD':    1,
    'TMA':  -1,   # alias: some exports use TMA rather than TMAT
    'TMB':  -1,
    'MP':    1,
    'PAL':   1,
    'PV':    1,
    'TR':    1,
    'BL':    1,
}

# Fail loudly rather than silently defaulting a direction to +1
missing = [t for t in tests if t not in test_direction]
assert not missing, f"No direction defined for: {missing}"
print("Tests:", list(tests))
print("Columns:", list(data.columns))

# passed to R so the sign flip happens inside the model fit loop
dir_df = pd.DataFrame({'test_name': list(tests),
                       'direction': [test_direction[t] for t in tests]})


In [ ]:
#BUILD THE COMPOSITE COGNITION MEASURE

"""
Composite = mean of the direction-corrected z-scores across tests at a visit,
re-standardised to SD 1 within the analysis sample.

Three things this has to get right:

1. DIRECTION BEFORE AVERAGING. z_score is a normative z on each test's own
   scale, so for timed tests (RT, TMA, TMB, PMat) a high z means slow and
   therefore worse, while for FI/NM/SD a high z means better. Averaging raw z
   across both families would cancel signal. z_worse flips the higher-is-better
   tests so that positive always means worse performance.

2. RE-STANDARDISATION. The mean of k correlated z-scores has SD well below 1
   (roughly 1/sqrt(k) if independent, more if correlated). Without rescaling,
   "per 1 SD" on the composite is not the same unit as "per 1 SD" on a single
   test and the ORs are not comparable. We divide by the observed SD of the
   composite in this cohort, so the composite OR reads per 1 SD of composite.

3. VISIT DEFINITION. Tests taken at the same session must group together.
   VISIT_KEYS is auto-detected; check the printed diagnostic and set it
   manually if the detection is wrong for your export.
"""

MIN_TESTS = 1          # visits with fewer contributing tests are dropped
COMPOSITE_NAME = 'COMPOSITE'

# ---- work out what defines a visit -------------------------------------
CANDIDATES = ['instance', 'instance_index', 'visit', 'wave', 'session',
              'assessment_instance', 'timepoint']
found = [c for c in CANDIDATES if c in data.columns]
if found:
    VISIT_KEYS = ['eid', found[0]]
    print(f"Visit key: {VISIT_KEYS}")
else:
    # fall back to rounding age at test — tests in one session share an age
    data['visit_age'] = data['age_at_test'].round(0)
    VISIT_KEYS = ['eid', 'visit_age']
    print("No instance/visit column found; grouping on eid + rounded age_at_test.")
    print("Check the tests-per-visit table below before trusting this.")

# ---- direction-correct, then average ------------------------------------
cd = data.dropna(subset=['z_score', 'age_at_test', 'dem_all']).copy()
cd['sign'] = cd['test_name'].map(test_direction).map({1: -1, -1: 1})
cd['z_worse'] = cd['sign'] * cd['z_score']

comp_data = (cd.groupby(VISIT_KEYS)
               .agg(z_raw=('z_worse', 'mean'),
                    n_tests=('z_worse', 'size'),
                    age_at_test=('age_at_test', 'mean'),
                    dem_all=('dem_all', 'first'))
               .reset_index())

print("\nTests contributing per visit:")
print(comp_data['n_tests'].value_counts().sort_index().to_string())

# A visit cannot contain more tests than exist. If it does, the visit key is
# collapsing two sessions into one - fix VISIT_KEYS before going further.
over = (comp_data['n_tests'] > len(tests)).sum()
assert over == 0, (
    f"{over} visits have more than {len(tests)} tests - VISIT_KEYS is merging "
    "separate sessions. Set VISIT_KEYS explicitly.")

comp_data = comp_data[comp_data['n_tests'] >= MIN_TESTS].copy()

COMP_SD = comp_data['z_raw'].std(ddof=1)
comp_data['z_score'] = comp_data['z_raw'] / COMP_SD
comp_data['test_name'] = COMPOSITE_NAME

print(f"\nComposite: {comp_data['eid'].nunique()} people, {len(comp_data)} visits, "
      f"{comp_data.groupby('eid')['dem_all'].first().sum()} with incident dementia")
print(f"Raw composite SD = {COMP_SD:.3f} (divided out; composite now has SD 1)")
print(f"Mean tests per visit = {comp_data['n_tests'].mean():.2f}")

# already oriented so that positive = worse, so no further flip in R
comp_data = comp_data[['eid', 'test_name', 'z_score', 'age_at_test',
                       'dem_all', 'n_tests']]
comp_data.to_csv(f'{output_root}/composite_input.csv', index=False)


In [ ]:
#Analysis loop 1/3
"""
This block uses %%R magic to loop through each cognitive test
Fits separate GEE model for each test to examine test-specific effects
-i data -i tests: imports Python objects to R
-o all_results: exports results back to Python

For each test:
1. Filters data to that test only
2. Fits GEE model: dem_all ~ z_score + age_at_test
3. Extracts coefficients with confidence intervals
4. Saves the robust covariance matrix (needed for age-equivalence CIs)
5. Computes age-equivalence: beta_cognition / beta_age, delta-method CI
6. Generates predictions
7. Saves results to test-specific subdirectory under output_root
"""
%load_ext rpy2.ipython


In [ ]:
%%R -i data -i tests -i dir_df -i output_root -o all_results

library(geepack)
library(broom)
library(dplyr)

# Shared fitter: returns a one-row summary for any dataframe with
# z_score / age_at_test / dem_all / eid. s = +1 if z_score already reads
# "higher is worse", -1 if it needs flipping.
fit_one <- function(test_df, label, s, output_root) {

  #skip a test if event number are too low
  n_events <- sum(tapply(test_df$dem_all, test_df$eid, function(x) x[1]))
  if (n_events < 25) {
    cat("  SKIPPING", label, "-", n_events, "cases at person level\n")
    return(NULL)
  }
      
  fit <- geeglm(dem_all ~ z_score + age_at_test,
                data = test_df,
                id = eid,
                family = binomial(link = "logit"),
                corstr = "independence",
                control = geese.control(maxit = 100))

  cat("  alpha =", fit$geese$alpha, " converged:", fit$geese$error == 0, "\n")
  cat("  error code:", fit$geese$error, "\n")
      
  tidy_results <- tidy(fit, exponentiate = FALSE)
  if (!"conf.low" %in% names(tidy_results)) {
    tidy_results$conf.low  <- tidy_results$estimate - 1.96 * tidy_results$std.error
    tidy_results$conf.high <- tidy_results$estimate + 1.96 * tidy_results$std.error
  }

  # ---- robust (sandwich) covariance matrix -------------------------------
  # geeglm stores it in fit$geese$vbeta, unnamed; label it from coef()
  b <- coef(fit)
  V <- fit$geese$vbeta
  dimnames(V) <- list(names(b), names(b))

  # ---- age-equivalence ---------------------------------------------------
  # ratio = beta_worse / beta_age = years of ageing carrying the same change
  # in log-odds of dementia as 1 SD worse performance.
  bz  <- s * b[["z_score"]]
  ba  <- b[["age_at_test"]]
  idx <- c("z_score", "age_at_test")

  Vsub <- V[idx, idx]
  Vsub[1, 2] <- s * Vsub[1, 2]        # covariance picks up the sign flip
  Vsub[2, 1] <- s * Vsub[2, 1]        # var(bz) is unchanged since s^2 = 1

  ratio    <- bz / ba
  g        <- c(1 / ba, -bz / ba^2)   # gradient of ratio wrt (bz, ba)
  ratio_se <- sqrt(as.numeric(t(g) %*% Vsub %*% g))
  age_t    <- ba / sqrt(V["age_at_test", "age_at_test"])

  out_dir <- file.path(output_root, label)
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(tidy_results, file.path(out_dir, 'coefficients.csv'), row.names = FALSE)
  write.csv(V,            file.path(out_dir, 'vcov.csv'))

  fit_age <- geeglm(dem_all ~ age_at_test,
                    data = test_df, id = eid,
                    family = binomial(link = "logit"),
                    corstr = "independence",
                    control = geese.control(maxit = 100))

  pred_df <- data.frame(
    eid = test_df$eid,
    predicted_prob = predict(fit, type = "response"),
    predicted_prob_age = predict(fit_age, type = "response"),
    age_at_test        = test_df$age_at_test,
    z_worse            = s * test_df$z_score,
    actual = test_df$dem_all
  )
  write.csv(pred_df, file.path(out_dir, 'predictions.csv'), row.names = FALSE)

  z_coef <- tidy_results[tidy_results$term == "z_score", ]
  if (nrow(z_coef) == 0) return(NULL)

  data.frame(
    test = label,
    n_participants = length(unique(test_df$eid)),
    n_observations = nrow(test_df),
    z_estimate = z_coef$estimate,
    z_se       = z_coef$std.error,
    z_or       = exp(z_coef$estimate),
    z_ci_low   = exp(z_coef$conf.low),
    z_ci_high  = exp(z_coef$conf.high),
    z_pval     = z_coef$p.value,
    sign_flip  = s,
    # sign-corrected: always "per 1 SD worse"
    z_or_adj      = exp(s * z_coef$estimate),
    z_ci_low_adj  = exp(s * ifelse(s == 1, z_coef$conf.low,  z_coef$conf.high)),
    z_ci_high_adj = exp(s * ifelse(s == 1, z_coef$conf.high, z_coef$conf.low)),
    # age model + covariance
    age_estimate = ba,
    age_se       = sqrt(V["age_at_test", "age_at_test"]),
    age_t        = age_t,
    cov_z_age    = V["z_score", "age_at_test"],
    # age-equivalence (years)
    age_equiv    = ratio,
    age_equiv_se = ratio_se,
    age_equiv_lo = ratio - 1.96 * ratio_se,
    age_equiv_hi = ratio + 1.96 * ratio_se
  )
}

all_results <- list()

for (test in tests) {
  cat("\nAnalyzing", test, "\n")

  test_df <- data %>%
    filter(test_name == test) %>%
    filter(!is.na(z_score), !is.na(age_at_test), !is.na(dem_all))

  cat("  N participants:", length(unique(test_df$eid)), "\n")
  cat("  N observations:", nrow(test_df), "\n")

  s <- ifelse(dir_df$direction[dir_df$test_name == test] == 1, -1, 1)
  res <- fit_one(test_df, test, s, output_root)

  if (!is.null(res)) {
    all_results[[test]] <- res
    cat("  OR per 1 SD worse:", round(res$z_or_adj, 3),
        " | age-equivalent:", round(res$age_equiv, 2), "years\n")
  }
}

all_results <- do.call(rbind, all_results)


In [ ]:
%%R -i comp_data -i output_root -o comp_result

# Composite fit. comp_data$z_score is already oriented so that higher = worse
# and already standardised to SD 1, so s = 1 (no flip).

comp_df <- comp_data %>%
  filter(!is.na(z_score), !is.na(age_at_test), !is.na(dem_all))

cat("\nAnalyzing COMPOSITE\n")
cat("  N participants:", length(unique(comp_df$eid)), "\n")
cat("  N visits:", nrow(comp_df), "\n")

comp_result <- fit_one(comp_df, "COMPOSITE", 1, output_root)

cat("  OR per 1 SD worse:", round(comp_result$z_or_adj, 3),
    " | age-equivalent:", round(comp_result$age_equiv, 2), "years\n")


In [ ]:
#DISPLAY AND SAVE SUMMARY

"""
Prints summary table showing results from all tests plus the composite
Saves combined results to CSV in the group's own directory
"""

all_results = pd.concat([all_results, comp_result], ignore_index=True)
all_results['is_composite'] = all_results['test'].eq('COMPOSITE')

print("\nAll tests complete")
cols = ['test', 'n_participants', 'z_or_adj', 'z_ci_low_adj', 'z_ci_high_adj',
        'z_pval', 'age_equiv', 'age_equiv_lo', 'age_equiv_hi']
print(all_results.sort_values(['is_composite', 'z_or_adj'], ascending=[True, False])[cols]
      .to_string(index=False))
all_results.to_csv(f'{output_root}/summary.csv', index=False)

# sanity check: age must be a strong, positive predictor for the
# age-equivalence denominator to be stable
bad_age = all_results[all_results['age_t'].abs() < 10]
if len(bad_age):
    print("\nWARNING - weak age coefficient, age-equivalence CI unreliable for:")
    print(bad_age[['test', 'age_estimate', 'age_t']].to_string(index=False))


In [ ]:
#refresher block
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'black'
plt.rcParams['axes.linewidth'] = 0.8

GROUP = globals().get('GROUP', 'all')   # inherits from cell 0; only defaults on a fresh kernel
output_root = 'geeglm_pertest' if GROUP == 'all' else f'geeglm_pertest_{GROUP}'
os.makedirs(output_root, exist_ok=True)

all_results = pd.read_csv(f'{output_root}/summary.csv')
all_results['is_composite'] = all_results['test'].eq('COMPOSITE')
series = all_results['test'].tolist()          # includes COMPOSITE
tests = [t for t in series if t != 'COMPOSITE']


In [ ]:
#FOREST PLOT OF ODDS RATIOS

"""
Creates forest plot showing OR with 95% CI for each cognitive test, with the
composite drawn separately at the top. Direction adjustment happens in R, so
z_or_adj is read straight from summary.csv
"""

per_test = all_results[~all_results['is_composite']].sort_values('z_or_adj').reset_index(drop=True)
comp_row = all_results[all_results['is_composite']]

plt.figure(figsize=(10, 8))
y_pos = np.arange(len(per_test))

plt.scatter(per_test['z_or_adj'], y_pos, s=100, zorder=3, color='black')
for i, row in per_test.iterrows():
    plt.plot([row['z_ci_low_adj'], row['z_ci_high_adj']], [i, i], 'k-', linewidth=2, zorder=2)

labels = list(per_test['test'])
if len(comp_row):
    r = comp_row.iloc[0]
    yc = len(per_test) + 0.6
    plt.axhline(len(per_test) - 0.3, color='0.6', lw=1)
    plt.plot([r['z_ci_low_adj'], r['z_ci_high_adj']], [yc, yc], '-', color='0.15', linewidth=3, zorder=2)
    plt.scatter([r['z_or_adj']], [yc], s=140, marker='D', color='0.15', zorder=3)
    y_pos = np.append(y_pos, yc)
    labels.append('COMPOSITE')

plt.axvline(x=1, color='darkgrey', linestyle='--', linewidth=1)
plt.yticks(y_pos, labels)
plt.xlabel('Odds Ratio per 1SD reduction in test performance', fontsize=12)
plt.ylabel('Cognitive Test', fontsize=12)
plt.title(f'Association Between Cognitive Performance and\nOdds Ratio of Dementia by Test Type ({GROUP})',
          fontsize=14)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{output_root}/forest_plot.png', dpi=800, bbox_inches='tight')
plt.show()


In [ ]:
#DISCRIMINATION: INCREMENTAL AUC OVER AGE + AGE-ADJUSTED AUC
"""
Person-level, from the GEE fitted probabilities (both models fitted in R on
identical rows):
  auc_age   : age-only GEE
  auc       : age + test GEE
  delta_auc : auc - auc_age, paired person-level bootstrap 95% CI
  auc_adj   : age-stratified AUC of the test score alone = probability that a
              future case scored worse than a non-case in the same
              AGE_BAND-year age band (pooled Mann-Whitney within bands)
Sensitivity: auc_adj point estimates at SENS_BANDS widths.
Apparent (in-sample) estimates; bootstrap holds fitted values fixed.
"""
from sklearn.metrics import roc_auc_score

N_BOOT     = 1000
AGE_BAND   = 1          # years, primary
SENS_BANDS = [2, 5]     # sensitivity, point estimates only
RNG = np.random.default_rng(0)
ci = lambda x: np.nanpercentile(x, [2.5, 97.5])


def stratified_auc(y, score, band):
    """Pooled within-band concordance (ties count 0.5)."""
    df = pd.DataFrame({'y': y, 's': score, 'b': band})
    df['r'] = df.groupby('b')['s'].rank()
    g = df.groupby('b')['y'].agg(n1='sum', n='size')
    g['rs'] = df[df['y'] == 1].groupby('b')['r'].sum()
    g = g.fillna(0)
    g['n0'] = g['n'] - g['n1']
    g = g[(g['n1'] > 0) & (g['n0'] > 0)]
    u = g['rs'] - g['n1'] * (g['n1'] + 1) / 2
    return u.sum() / (g['n1'] * g['n0']).sum()


disc_results = []
for name in series:
    pred_file = f'{output_root}/{name}/predictions.csv'
    if not os.path.exists(pred_file):
        print(f'missing predictions for {name} - skipping'); continue
    pp = (pd.read_csv(pred_file).groupby('eid')
            .agg(y=('actual', 'first'), pf=('predicted_prob', 'mean'),
                 pa=('predicted_prob_age', 'mean'),
                 age=('age_at_test', 'mean'), zw=('z_worse', 'mean')))
    y  = pp['y'].to_numpy(int)
    pf, pa = pp['pf'].to_numpy(), pp['pa'].to_numpy()
    zw, age = pp['zw'].to_numpy(), pp['age'].to_numpy()
    band = np.floor(age / AGE_BAND).astype(int)

    auc, auc_age = roc_auc_score(y, pf), roc_auc_score(y, pa)
    auc_adj = stratified_auc(y, zw, band)
    sens = {f'auc_adj_{w}y': stratified_auc(y, zw, np.floor(age / w).astype(int))
            for w in SENS_BANDS}

    bf, ba, bj = (np.full(N_BOOT, np.nan) for _ in range(3))
    for b in range(N_BOOT):
        i = RNG.integers(0, len(y), len(y))
        if y[i].min() == y[i].max(): continue
        bf[b] = roc_auc_score(y[i], pf[i])
        ba[b] = roc_auc_score(y[i], pa[i])
        bj[b] = stratified_auc(y[i], zw[i], band[i])

    disc_results.append({
        'test': name, 'n_people': len(y), 'n_dementia': int(y.sum()),
        'prevalence': y.mean(),
        'auc_age': auc_age, 'auc_age_lo': ci(ba)[0], 'auc_age_hi': ci(ba)[1],
        'auc': auc, 'auc_lo': ci(bf)[0], 'auc_hi': ci(bf)[1],
        'delta_auc': auc - auc_age,
        'delta_auc_lo': ci(bf - ba)[0], 'delta_auc_hi': ci(bf - ba)[1],
        'auc_adj': auc_adj, 'auc_adj_lo': ci(bj)[0], 'auc_adj_hi': ci(bj)[1],
        **sens})
    print(f"{name:10s} age {auc_age:.3f}  full {auc:.3f}  dAUC {auc-auc_age:+.3f}  "
          f"adj {auc_adj:.3f}  " +
          '  '.join(f"{k[8:]} {v:.3f}" for k, v in sens.items()))

disc_df = pd.DataFrame(disc_results)
disc_df['is_composite'] = disc_df['test'].eq('COMPOSITE')
disc_df = disc_df.sort_values(['is_composite', 'delta_auc'], ascending=[True, False])
print('\nDISCRIMINATION BY TEST')
print(disc_df.round(3).to_string(index=False))
disc_df.to_csv(f'{output_root}/discrimination_summary.csv', index=False)
disc_df[['test', 'auc', 'n_people']].to_csv(f'{output_root}/auc_summary.csv', index=False)

In [ ]:
#CALCULATE PERFORMANCE METRICS FOR EACH TEST

"""
For each test and the composite, calculates comprehensive performance metrics:
AUC, optimal threshold, sensitivity, specificity, PPV, NPV
Uses Youden's J statistic to find optimal threshold

NOTE: the Youden threshold is chosen on the same data it is evaluated on, so
sensitivity/specificity here are optimistic. Treat as descriptive only.
"""

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

performance_results = []

for name in series:
    pred_file = f'{output_root}/{name}/predictions.csv'
    if not os.path.exists(pred_file):
        continue
    pred_df = pd.read_csv(pred_file)

    person_preds = pred_df.groupby('eid').agg(
        predicted_prob=('predicted_prob', 'mean'),
        actual=('actual', 'first')
    ).reset_index()

    auc = roc_auc_score(person_preds['actual'], person_preds['predicted_prob'])

    fpr, tpr, thresholds = roc_curve(person_preds['actual'], person_preds['predicted_prob'])
    youden_j = tpr - fpr
    optimal_idx = np.argmax(youden_j)
    optimal_threshold = thresholds[optimal_idx]

    person_preds['pred_class'] = (person_preds['predicted_prob'] > optimal_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        person_preds['actual'], person_preds['pred_class'], labels=[0, 1]).ravel()

    performance_results.append({
        'test': name,
        'n_people': len(person_preds),
        'n_dementia': int(person_preds['actual'].sum()),
        'auc': auc,
        'optimal_threshold': optimal_threshold,
        'sensitivity': tp / (tp + fn),
        'specificity': tn / (tn + fp),
        'ppv': tp / (tp + fp) if (tp + fp) else np.nan,
        'npv': tn / (tn + fn) if (tn + fn) else np.nan,
        'true_negatives': tn,
        'false_positives': fp,
        'false_negatives': fn,
        'true_positives': tp
    })

performance_df = pd.DataFrame(performance_results).sort_values('auc', ascending=False)
print("MODEL PERFORMANCE BY TEST \n")
print(performance_df.to_string(index=False))
performance_df.to_csv(f'{output_root}/performance_summary.csv', index=False)


In [ ]:
#MERGE INTO A SINGLE EXTENDED SUMMARY

"""
Joins the GEE effect sizes, age-equivalence and discrimination metrics into one
file per cohort, composite included. This is what the visualisation notebook
should read - it replaces the hardcoded POOLED dict in the figure cells.

Also emits a publication-ready CSV with everything rounded and formatted,
composite last.
"""

ext = (all_results
       .merge(disc_df[['test', 'auc_age', 'auc', 'auc_lo', 'auc_hi',
                'delta_auc', 'delta_auc_lo', 'delta_auc_hi',
                'auc_adj', 'auc_adj_lo', 'auc_adj_hi',
                'prevalence', 'n_dementia']], on='test', how='left'))
ext['group'] = GROUP
ext.to_csv(f'{output_root}/summary_extended.csv', index=False)
print(f'Saved: {output_root}/summary_extended.csv')

# ---- publication table --------------------------------------------------
LABELS = {'FI':'Fluid intelligence','MP':'Matrix pattern','NM':'Numeric memory',
          'PAL':'Paired associate learning','PMat':'Pairs matching',
          'PV':'Picture vocabulary','RT':'Reaction time','SD':'Symbol digit',
          'TMA':'Trail making A','TMAT':'Trail making A','TMB':'Trail making B',
          'TR':'Tower rearranging','BL':'Broken letter','COMPOSITE':'Composite'}

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def ci(est, lo, hi, dp=2):
    return f'{est:.{dp}f} ({lo:.{dp}f}\u2013{hi:.{dp}f})'

tbl = ext.sort_values(['is_composite', 'z_or_adj'], ascending=[True, False])
pub = pd.DataFrame({
    'Cognitive test':        tbl['test'].map(lambda t: LABELS.get(t, t)),
    'Participants, n':       tbl['n_participants'].map('{:,}'.format),
    'Observations, n':       tbl['n_observations'].map('{:,}'.format),
    'Incident dementia, n (%)': [f'{int(n):,} ({pv*100:.1f})'
                                 for n, pv in zip(tbl['n_dementia'], tbl['prevalence'])],
    'OR per 1 SD worse (95% CI)': [ci(a, b, c) for a, b, c in
                                   zip(tbl['z_or_adj'], tbl['z_ci_low_adj'], tbl['z_ci_high_adj'])],
    'P value':               tbl['z_pval'].map(fmt_p),
    'Age equivalent, years (95% CI)': [ci(a, b, c, dp=1) for a, b, c in
                                       zip(tbl['age_equiv'], tbl['age_equiv_lo'], tbl['age_equiv_hi'])],
    'AUC, age only':               tbl['auc_age'].map('{:.3f}'.format),
    'AUC, age + test (95% CI)':    [ci(a, b, c, dp=3) for a, b, c in zip(tbl['auc'], tbl['auc_lo'], tbl['auc_hi'])],
    'ΔAUC (95% CI)':               [ci(a, b, c, dp=3) for a, b, c in zip(tbl['delta_auc'], tbl['delta_auc_lo'], tbl['delta_auc_hi'])],
    'Age-adjusted AUC (95% CI)':   [ci(a, b, c, dp=2) for a, b, c in zip(tbl['auc_adj'], tbl['auc_adj_lo'], tbl['auc_adj_hi'])],
})

pub['Cognitive test'] = [f'{n}*' if c < 100 else n
                         for n, c in zip(pub['Cognitive test'], tbl['n_dementia'])]

pub.to_csv(f'{output_root}/table_publication_{GROUP}.csv', index=False)
print(f'Saved: {output_root}/table_publication_{GROUP}.csv\n')
print(pub.to_string(index=False))

FOOTNOTE += (" Tests marked * had fewer than 100 incident dementia cases; "
             "their estimates are correspondingly imprecise.")

print('\nTABLE FOOTNOTE\n' + FOOTNOTE)

# paste-ready replacement for POOLED in the visualisation notebook
c = ext[ext['is_composite']].iloc[0]
print(f"\nPOOLED['{GROUP}'] = ({c['z_or_adj']:.3f}, {c['z_ci_low_adj']:.3f}, "
      f"{c['z_ci_high_adj']:.3f}, {c['z_pval']:.2g})")
